In [ ]:
#set up

In [ ]:
from datetime import datetime
start = datetime.now()

In [ ]:
import os
bucket = os.getenv("WORKSPACE_BUCKET")
bucket

In [ ]:
#Read All of Us srWGS Hail VDS

In [ ]:
##1 Import Hail and Initialize Spark

In [ ]:
import hail as hl
hl.default_reference(new_default_reference = "GRCh38")

In [ ]:
##2 Read Hail VDS

In [ ]:
vds_srwgs_path = os.getenv("WGS_VDS_PATH")
vds_srwgs_path

In [ ]:
# hardcode the VDS path in case you are not in a v8 workspace bucket
vds_srwgs_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/vds/hail.vds"

In [ ]:
vds = hl.vds.read_vds(vds_srwgs_path)

In [ ]:
#filter out IG loci interval , 1 MB flanks 

In [ ]:
vds = hl.vds.filter_chromosomes(vds, keep= ["chr2", "chr14", "chr22"])

In [ ]:
test_intervals = ['chr2:87857361-91235368', 'chr14:104586437-107043718', 'chr22:21026076-23922913' ]

In [ ]:
vds = hl.vds.filter_intervals(
    vds,
    [hl.parse_locus_interval(x,)
     for x in test_intervals])


In [ ]:
##get the aggregated counts of fields in the variant data.

In [ ]:
##The All of Us VDS uses Local Alleles (LA) which has information for reference alleles and all alternative 
##alleles across all samples in the array. We will need to convert Local Alleles related fields, 
##Local Allele Depth (LAD) and Local Genotypte (LGT) to the Global Allele format fields (AD and GT), 
##which have information for the reference allele and alternative allele for each sample in the array. 
##We will use two diferent functions to convert LAD to AD and LGT to GT.

In [ ]:
mt = vds.variant_data.annotate_entries(AD = hl.vds.local_to_global(vds.variant_data.LAD, 
                                                                   vds.variant_data.LA, 
                                                                   n_alleles=hl.len(vds.variant_data.alleles), 
                                                                   fill_value=0, number='R'))

In [ ]:
mt = mt.annotate_entries(GT = hl.vds.lgt_to_gt(mt.LGT, mt.LA))

In [ ]:
##The field FT is boolean, which is not accepted by VCF. We will tranform it to "Fail" or "PASS", using the function transmute_entry.

In [ ]:
mt = mt.transmute_entries(FT = hl.if_else(mt.FT, "PASS", "FAIL"))

In [ ]:
##Densify the MatrixTable
###This step is necessary before performing any dense MatrixTable-related computations, 
###such as computing AC / AF / AN, variant QC, or converting to VCF / PLINK / BGEN files, etc.

In [ ]:
mt = hl.vds.to_dense_mt(hl.vds.VariantDataset(vds.reference_data, mt))

In [ ]:
##Create fields AC / AF / AN
##There are no Allele Counts (AC) / Allele Frequency (AF) / Allele Number (AN) in the VDS. We will use the function agg.call_stats to get the AC / AF / AN.

In [ ]:
mt = mt.annotate_rows(info = hl.agg.call_stats(mt.GT, mt.alleles))

In [ ]:
##Remove unneccessary fields
##We will use the function drop to drop fields.



In [ ]:
fields_to_drop_list = ['as_vets','as_vqsr','LAD', 'LGT', 'LA',
            'tranche_data', 'truth_sensitivity_snp_threshold', 
             'truth_sensitivity_indel_threshold','snp_vqslod_threshold','indel_vqslod_threshold']

In [ ]:
mt = mt.drop(*(f for f in fields_to_drop_list if f in mt.entry or f in mt.row or f in mt.col or f in mt.globals))


In [ ]:
mt.describe()

In [ ]:
#Write the MatrixTable into your bucket for downstream analysis.

out_path = f'{bucket}/data/phewas_hail_finally.mt'
mt.write(out_path, overwrite = True)

In [ ]:
!gsutil ls -l {bucket}/data/

In [ ]:
# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'test.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
my_dataframe = pd.read_csv(name_of_file_in_bucket)
my_dataframe.head()


In [ ]:
##Convert to dense PLINK BED files

In [ ]:
#Before converting to PLINK files, we need to split multi-allelic variants using the function split_multi.

mt_plink = hl.split_multi_hts(mt)

In [ ]:
##We use the function export_plink to create a PLINK BED triplet. It will write three files into the specified output path: .bed, .bim, .fam.

##The VDS contains some phased genotypes, and since PLINK BED files do not support phased data, please use the function below to unphase the data before exporting to PLINK BED files—especially when working with sex chromosomes.

In [ ]:
def unphasing_mt(mt):
    mt = mt.select_entries(GT = hl.unphased_diploid_gt_index_call(mt.GT.n_alt_alleles()))
    return mt



In [ ]:
out_path = f'{bucket}/data/phewas_plink_finally'

In [ ]:
hl.export_plink(mt_plink, out_path, ind_id = mt_plink.s)

In [ ]:
#import hail as hl
#import os

# Make sure Hail is initialized
#hl.init() 

bucket = os.getenv("WORKSPACE_BUCKET")
gcs_plink_prefix = f'{bucket}/data/phewas_plink_finally'

print("Attempting to read PLINK files back into Hail from GCS...")

try:
    # Hail reads the .bed, .bim, and .fam files directly from GCS
    mt_check = hl.import_plink(
        bed=f'{gcs_plink_prefix}.bed',
        bim=f'{gcs_plink_prefix}.bim',
        fam=f'{gcs_plink_prefix}.fam'
    )
    
    # Run a simple action like .describe() to confirm it can be parsed
    mt_check.describe()
    
    print("\n--- CHECK SUCCESSFUL! --- ✅")
    print("Hail successfully read the PLINK files back from GCS.")
    print("This confirms the file integrity.")

except Exception as e:
    print("\n--- CHECK FAILED! --- ❌")
    print("Hail encountered an error trying to read the PLINK files:")
    print(e)